In [1]:
import pyspark
import pandas as pd

# Carga ufnciones extra
from pyspark.sql.functions import *
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('sesion_plantilla').getOrCreate()
spark.range(5).show()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 15:37:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [ ]:
from pyspark.sql.functions import explode, col, lit
from functools import reduce
import os

countries = ["US", "GB", "DE", "CA", "FR", "RU", "MX", "KR", "JP", "IN"]

data_dict = {}
cat_dict = {}
joined_tables = []

for country in countries:
    
    # -----------------------------
    # Load CSV
    # -----------------------------
    data_path = f"Data/{country}videos.csv"
    
    data_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(data_path)
        .withColumn("category_id", col("category_id").cast("string"))
    )
    
    # -----------------------------
    # Load JSON
    # -----------------------------
    cat_path = f"Data/{country}_category_id.json"
    
    cat_raw = (
        spark.read
        .option("multiline", "true")
        .json(cat_path)
    )
    
    cat_df = (
        cat_raw
        .select(explode("items").alias("item"))
        .select(
            col("item.id").alias("category_id"),
            col("item.snippet.title").alias("category_name")
        )
        .withColumn("category_id", col("category_id").cast("string"))
    )
    
    # Guardar en diccionarios
    data_dict[country] = data_df
    cat_dict[country] = cat_df
    
    # -----------------------------
    # Join
    # -----------------------------
    df_joined = (
        data_df
        .join(cat_df, on="category_id", how="left")
        .withColumn("country", lit(country))
    )
    
    joined_tables.append(df_joined)

# -----------------------------
# Union final
# -----------------------------
youtube_full = reduce(lambda df1, df2: df1.unionByName(df2), joined_tables)

print("Tabla unificada creada profesionalmente: youtube_full")

✅ Tabla unificada creada profesionalmente: youtube_full


In [5]:
#Revisando la data
youtube_full.show(5)

+-----------+-----------+-------------+--------------------+--------------------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+--------------+-------+
|category_id|   video_id|trending_date|               title|       channel_title|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description| category_name|country|
+-----------+-----------+-------------+--------------------+--------------------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+--------------+-------+
|         22|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    296

In [6]:
youtube_full.printSchema()

root
 |-- category_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: long (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: boolean (nullable = true)
 |-- ratings_disabled: boolean (nullable = true)
 |-- video_error_or_removed: boolean (nullable = true)
 |-- description: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- country: string (nullable = false)



In [7]:
#Filtrar valores NO númericos
youtube_clean = youtube_full.filter(
    col("views").rlike("^[0-9]+$") &
    col("likes").rlike("^[0-9]+$") &
    col("dislikes").rlike("^[0-9]+$") &
    col("comment_count").rlike("^[0-9]+$")
)

In [8]:
#Columnas a enteros
youtube_clean = youtube_clean \
.withColumn("views", col("views").cast("int")) \
.withColumn("likes", col("likes").cast("int")) \
.withColumn("dislikes", col("dislikes").cast("int")) \
.withColumn("comment_count", col("comment_count").cast("int"))

In [10]:
# Calcular estadísticas por región y categoría
# Promedio, máximo y mínimo de vistas, likes, dislikes y comentarios.
# Mostrar resultados ordenados por país y categoría.   

stats_region_category = (
    youtube_clean
    .groupBy("country", "category_id")
    .agg(
        avg("views").alias("avg_views"),
        max("views").alias("max_views"),
        min("views").alias("min_views"),
        
        avg("likes").alias("avg_likes"),
        max("likes").alias("max_likes"),
        min("likes").alias("min_likes"),
        
        avg("dislikes").alias("avg_dislikes"),
        max("dislikes").alias("max_dislikes"),
        min("dislikes").alias("min_dislikes"),
        
        avg("comment_count").alias("avg_comments")
    )
)

stats_region_category.orderBy("max_views", ascending=False).show(20)

+-------+-----------+--------------------+---------+---------+------------------+---------+---------+------------------+------------+------------+------------------+
|country|category_id|           avg_views|max_views|min_views|         avg_likes|max_likes|min_likes|      avg_dislikes|max_dislikes|min_dislikes|      avg_comments|
+-------+-----------+--------------------+---------+---------+------------------+---------+---------+------------------+------------+------------+------------------+
|     GB|         10|1.2444442690780863E7|424538912|     2152|272138.50894285296|  5613827|        0|11587.191798749454|      421473|           0|21303.849062091027|
|     US|         10|   6201003.119592089|225211923|     1591|218918.19901112485|  5613827|        0| 7907.757725587145|      343541|           0|19359.764524103834|
|     GB|         24|  3264607.9615300307|169884583|     2650| 81572.36201227532|  3312868|        0|  9656.52685225778|     1944971|           0|12812.082419991231|
|   

In [11]:
#calculando la participación de cada país en el total de vistas
total_views = youtube_clean.agg(sum("views")).collect()[0][0]
participacion_pais = (
    youtube_clean
    .groupBy("country")
    .agg(sum("views").alias("views_country"))
    .withColumn(
        "share_views",
        col("views_country") / total_views * 100
    )
)

participacion_pais.show()

+-------+-------------+------------------+
|country|views_country|       share_views|
+-------+-------------+------------------+
|     US|  96671770152|19.384262318685465|
|     GB| 230069198174| 46.13261639713733|
|     DE|  24645115205| 4.941746460800275|
|     CA|  46891975069|   9.4026037171923|
|     FR|  17100897444| 3.429008090140736|
|     RU|   9806494525|1.9663616586358752|
|     MX|  13849692994| 2.777088715835437|
|     KR|  14689152313|2.9454139634209073|
|     JP|   5377457949|1.0782677851788542|
|     IN|  39610905768| 7.942630892972821|
+-------+-------------+------------------+



In [14]:
#calculando la participación de cada categoría en el total de vistas
total_views = youtube_clean.agg(sum("views")).collect()[0][0]

participacion_categoria = (
    youtube_clean
    .groupBy("category_id")
    .agg(sum("views").alias("views_region_cat"))
    .withColumn(
        "share_global",
        col("views_region_cat") / total_views * 100
    )
)

participacion_categoria.orderBy("share_global", ascending=False).show(20)

+-----------+----------------+--------------------+
|category_id|views_region_cat|        share_global|
+-----------+----------------+--------------------+
|         10|    255967088943|   51.32556473539193|
|         24|    104517467253|  20.957452200691442|
|          1|     27619339220|   5.538126752695666|
|         22|     23600365409|   4.732257133448405|
|         23|     22050866339|   4.421557366720095|
|         17|     18972425164|  3.8042798391128505|
|         25|     10422447730|  2.0898702949521617|
|         26|      9771031927|  1.9592508309241938|
|         28|      9194715151|  1.8436899433240412|
|         20|      7730729502|  1.5501370084146366|
|         27|      2734841410|  0.5483801859435266|
|         15|      2008474231|  0.4027317519148438|
|          2|      1661853766|  0.3332287107682891|
|         29|      1219859213|    0.24460161368182|
|         19|       726674959| 0.14571014892484188|
|         43|       444064556| 0.08904216635735727|
|         30